# Incremental Ingestion (`ingest_nbk`)

Inject **new chunks + their precomputed embeddings** (already processed upstream, e.g. on HPC)
into the `optimized_*` family, then refresh the Vector Search index.

The upsert is idempotent (MERGE on `chunk_id`), so re-running with overlapping ids updates rows
in place rather than duplicating them.


# Ingest HH

In [0]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import settings
from ingest import load_new, ingest, sync_vector_index, rebuild_bm25
from lake_io import read_chunks_jsonl, read_embeddings_npy

In [0]:
base = settings.LAKE_BASE   # abfss://.../RAG_files/
display(dbutils.fs.ls(base))                            # find the optimized_HH_* dirs
display(dbutils.fs.ls(base + "optimized_HH_chunks/"))   # confirm */chunks.jsonl layout
# Find the HH embeddings dir, e.g.:
# display(dbutils.fs.ls(base + "optimized_HH_embeddings/"))
# display(dbutils.fs.ls(base + "optimized_HH_embeddings/DMRetriever-33M/"))

path,name,size,modificationTime
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/converted_raw_txts/,converted_raw_txts/,0,1772733625000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/,optimized_HH_chunks/,0,1770926796000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_embeddings/,optimized_HH_embeddings/,0,1770926786000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/,optimized_Qs_chunks/,0,1771264808000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_embeddings/,optimized_Qs_embeddings/,0,1771264803000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_chunks/,optimized_chunks/,0,1770260233000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_embeddings/,optimized_embeddings/,0,1770269731000


path,name,size,modificationTime
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/100% Big Sandy Creek Draft Milestone Report/,100% Big Sandy Creek Draft Milestone Report/,0,1770926795000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/100% Boggy Creek Draft Milestone Report/,100% Boggy Creek Draft Milestone Report/,0,1770926788000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/100% Cypress Creek Draft Milestone Report/,100% Cypress Creek Draft Milestone Report/,0,1770926793000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/100% LPIB Draft Milestone Report/,100% LPIB Draft Milestone Report/,0,1770926795000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/100% PIB Draft Milestone Report/,100% PIB Draft Milestone Report/,0,1770926795000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/1211020301_UpperLagunaMadre_100%_Final_Report/,1211020301_UpperLagunaMadre_100%_Final_Report/,0,1770926788000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/1211020301_UpperLagunaMadre_100%_Final_SlideDeck/,1211020301_UpperLagunaMadre_100%_Final_SlideDeck/,0,1770926795000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/2025-02-13 Quicksand Creek - Sabine River 100% Report_Signed/,2025-02-13 Quicksand Creek - Sabine River 100% Report_Signed/,0,1770926794000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/Adams Bayou 100% Final Submittal_Update0425_Final_Signed/,Adams Bayou 100% Final Submittal_Update0425_Final_Signed/,0,1770926793000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/Adlong Ditch-Ceder Bayou_Final/,Adlong Ditch-Ceder Bayou_Final/,0,1770926795000


In [0]:
base  = settings.LAKE_BASE
model = settings.EMBED_MODEL_NAME          # "DMRetriever-33M"

hh_chunks = read_chunks_jsonl(spark, base + "optimized_HH_chunks/*/chunks.jsonl")

# Set HH_EMB to the real HH embeddings prefix found in Cell 1
HH_EMB = base + "optimized_HH_embeddings/"
hh_emb = read_embeddings_npy(
    spark,
    npy_glob=HH_EMB + f"{model}/*/embeddings.npy",
    ids_glob=HH_EMB + f"{model}/*/chunk_ids.json",
    model_name=model,                       # used to pair npy/ids by source_dir; must match the path segment
)

print("HH chunks:", hh_chunks.count())
print("HH emb   :", hh_emb.count())
display(hh_chunks.limit(5))

HH chunks: 1368
HH emb   : 1368


chunk_id,source_file,chunk_index_in_file,text
"Dry Bayou, Austin, Lower Oyster GLO RBFS Baseline Conditions Report.txt_0","/net/cven-mosta-nas.engr.tamu.edu/volume1/NASdata1/TDIS_PDF/HHtxt/Dry Bayou, Austin, Lower Oyster GLO RBFS Baseline Conditions Report.txt",0,"Page 0 of 261 Dry Bayou-Austin Bayou- Baseline Conditions Contract No. 20-094-002, Deliverable 49 PREPARED FOR: Texas General Land Office Krystle Haney & Skyler LeVrier GLO-CDR Central Region PM 1700 N. Congress Avenue Austin, Texas 78701 SUBMITTED BY: Central Region Vendor PM Krista Melnar, PE, CFM, PMP 10431 Morado Circle, Bldg. 5 Suite 300 Austin, TX 78759 DRAFT THIS DOCUMENT IS RELEASED FOR THE PURPOSE OF INTERIM REVIEW UNDER THE AUTHORITY OF ANDREW ALLEN SWYNENBERG, P. E. , TEXAS NO. 138771 ON 1/27/2025. IT IS NOT TO BE USED FOR CONSTRUCTION, BIDDING OR PERMIT PURPOSES. FREESE AND NICHOLS, INC. TEXAS REGISTERED ENGINEERING FIRM F- 2144 The enclosed hydrologic and hydraulic models, final report, and GIS data were developed for the Texas General Land Office (GLO) for its Regional River Basin Flood Study project. The purpose of these models, final report, and associated GIS data is to better understand existing conditions for the GLO study. These products should not be used for construction or bidding purposes. For areas where FEMA effective FIRM maps do not include regulatory flood zones, or where Zone A is depicted, these maps could be considered best available data at the discretion of the local floodplain administrator. Enclosed models, reports, and GIS data may require additional effort to support a FEMA map update. The Texas General Land Office and Freese and Nichols, Inc. make no representations or warranties regarding the accuracy or completeness of the information provided in the enclosed documents or the data from which they were produced. No representations or warranties are made regarding the utility of any of the data or data representations for purposes other than those stated above. Not for use beyond those included in and specific to the scope of the RBFS project."
"Dry Bayou, Austin, Lower Oyster GLO RBFS Baseline Conditions Report.txt_1","/net/cven-mosta-nas.engr.tamu.edu/volume1/NASdata1/TDIS_PDF/HHtxt/Dry Bayou, Austin, Lower Oyster GLO RBFS Baseline Conditions Report.txt",1,"... ... ... ... ... ... .. 126 Table 6-4 Calibration Metrics ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... 128 Table 6-5 Statistical Model Performance Metrics ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... 128 Table 6-6: Hurricane Harvey Available Gage Data ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... 130 Page 7 of 260 Table 6-7: Simulated vs. Observed Discharge, USGS Gages – Hurricane Harvey ... ... ... ... ... ... ... ... ... ... ... 136 Table 6-8: Simulated vs. Observed Discharge Statistics – Hurricane Harvey ... ... ... ... ... ... ... ... ... ... ... ... ... . 136 Table 6-9: Simulated vs. Observed Stage, USGS Gages – Hurricane Harvey ... ... ... ... ... ... ... ... ... ... ... ... ... .. 136 Table 6-10: Simulated vs. Observed Discharge, USGS Gages – Hurricane Ike ... ... ... ... ... ... ... ... ... ... ... ... .. 137 Table 6-11: Hurricane Ike Available Gage Data ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... .. 138 Table 6-12: Ike Geometry Changes ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... .. 138 Table 6-13: Simulated vs. Observed Flow, USGS Gages – Hurricane Ike ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... ... 143 Table 6-14 Simulated vs. Observed Flow Statistics - Hurricane Ike ... ... ... ... ... ... ... ... ... ... ... ...

In [0]:
n_chunks = hh_chunks.select("chunk_id").distinct().count()
n_emb    = hh_emb.select("chunk_id").distinct().count()
n_both   = hh_chunks.join(hh_emb, "chunk_id", "inner").select("chunk_id").distinct().count()
print(f"chunks={n_chunks}  emb={n_emb}  matched={n_both}")
# matched should be close to chunks; a large gap means chunk_ids don't line up across the two folders

chunks=1368  emb=1368  matched=1368


In [0]:
report = ingest(
    spark, hh_chunks, hh_emb,
    sync_index=False,            # do not trigger the vector index sync yet
    update_keyword_index=True,   # incremental BM25 update (cheap)
)
print(report)

{'chunks_text': 1368, 'embeddings': 1368, 'rag_chunks': 1368, 'bm25': {'mode': 'incremental', 'affected_chunks': 1368, 'new_postings': 153787, 'df_terms_changed': 7344}}


# Ingest Qs

In [0]:
base = settings.LAKE_BASE   # abfss://.../RAG_files/
display(dbutils.fs.ls(base))                            # find the optimized_Qs_* dirs
display(dbutils.fs.ls(base + "optimized_Qs_chunks/"))   # confirm */chunks.jsonl layout
# Find the QS embeddings dir, e.g.:
# display(dbutils.fs.ls(base + "optimized_Qs_embeddings/"))
# display(dbutils.fs.ls(base + "optimized_Qs_embeddings/DMRetriever-33M/"))

path,name,size,modificationTime
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/converted_raw_txts/,converted_raw_txts/,0,1772733625000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_chunks/,optimized_HH_chunks/,0,1770926796000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_HH_embeddings/,optimized_HH_embeddings/,0,1770926786000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/,optimized_Qs_chunks/,0,1771264808000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_embeddings/,optimized_Qs_embeddings/,0,1771264803000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_chunks/,optimized_chunks/,0,1770260233000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_embeddings/,optimized_embeddings/,0,1770269731000


path,name,size,modificationTime
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/2021-Winter-Storm-Uri-AAR-Findings-Report/,2021-Winter-Storm-Uri-AAR-Findings-Report/,0,1771264807000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/2024_State_Flood_Plan_Volume_II/,2024_State_Flood_Plan_Volume_II/,0,1771264805000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/4a7faeefdc092cbbbd0f13abffd99a800/,4a7faeefdc092cbbbd0f13abffd99a800/,0,1771264808000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/A_TC_Colorado_River_Flooding_AAR_05202019/,A_TC_Colorado_River_Flooding_AAR_05202019/,0,1771264806000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/City-of-Austin-Emergency-Operations-Plan-Basic-Plan---June-2025-for-web/,City-of-Austin-Emergency-Operations-Plan-Basic-Plan---June-2025-for-web/,0,1771264804000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/Coastal-Texas-Brochure103023/,Coastal-Texas-Brochure103023/,0,1771264806000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/Executive Summary - Friday/,Executive Summary - Friday/,0,1771264804000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/House-Interim-Committee-on-The-Panhandle-Wildfires-Report/,House-Interim-Committee-on-The-Panhandle-Wildfires-Report/,0,1771264803000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/NBS-to-Flooding-in-the-Hill-Country/,NBS-to-Flooding-in-the-Hill-Country/,0,1771264806000
abfss://tdis-data-bronze@tdisproddatalakehouse.dfs.core.windows.net/RAG_files/optimized_Qs_chunks/SFP-2024/,SFP-2024/,0,1771264807000


In [0]:
from pyspark.sql import functions as F

base  = settings.LAKE_BASE
model = settings.EMBED_MODEL_NAME          # "DMRetriever-33M"

# Read QS chunks and keep only the file name (strip the long NAS path prefix)
qs_chunks = (
    read_chunks_jsonl(spark, base + "optimized_Qs_chunks/*/chunks.jsonl")
    .withColumn("source_file", F.regexp_replace("source_file", r"^.*/", ""))
)

# Set QS_EMB to the real QS embeddings prefix found in Cell 1
QS_EMB = base + "optimized_Qs_embeddings/"
qs_emb = read_embeddings_npy(
    spark,
    npy_glob=QS_EMB + f"{model}/*/embeddings.npy",
    ids_glob=QS_EMB + f"{model}/*/chunk_ids.json",
    model_name=model,                       # used to pair npy/ids by source_dir; must match the path segment
)

print("QS chunks:", qs_chunks.count())
print("QS emb   :", qs_emb.count())
display(qs_chunks.select("source_file").distinct().limit(20))

QS chunks: 1209
QS emb   : 1209


source_file
2024_State_Flood_Plan_Volume_II.txt
SFP-2024.txt
A_TC_Colorado_River_Flooding_AAR_05202019.txt
Winter-Storm-Uri-AAR-and-Improvement-Plan-Technical-Report.txt
2021-Winter-Storm-Uri-AAR-Findings-Report.txt
Coastal-Texas-Brochure103023.txt
Executive Summary - Friday.txt
NBS-to-Flooding-in-the-Hill-Country.txt
SFP-2024 (1).txt
House-Interim-Committee-on-The-Panhandle-Wildfires-Report.txt


In [0]:
n_chunks = qs_chunks.select("chunk_id").distinct().count()
n_emb    = qs_emb.select("chunk_id").distinct().count()
n_both   = qs_chunks.join(qs_emb, "chunk_id", "inner").select("chunk_id").distinct().count()
print(f"chunks={n_chunks}  emb={n_emb}  matched={n_both}")
# matched should be close to chunks; a large gap means chunk_ids don't line up across the two folders

chunks=1209  emb=1209  matched=1209


In [0]:
report = ingest(
    spark, qs_chunks, qs_emb,
    sync_index=False,            # do not trigger the vector index sync yet
    update_keyword_index=True,   # incremental BM25 update (cheap)
)
print(report)

{'chunks_text': 1209, 'embeddings': 1209, 'rag_chunks': 1209, 'bm25': {'mode': 'incremental', 'affected_chunks': 1209, 'new_postings': 151049, 'df_terms_changed': 13555}}
